# Twilight Imperium: baseline RAG vs. re-ranking

This notebook answers questions from the supplied rules PDF. It compares the same vector-search top 10 under two orderings: vector rank and cross-encoder rank. Edit the queries below and run cells in order. API answers require `GEMINI_API_KEY` in `.env`.

## 1. Load the document

PyMuPDF supplies text blocks with positions. The rules pages have two columns, so each page is read left column first, then right column, with blocks ordered top to bottom. The physical PDF page number is retained. Check the printed preview for reading-order problems.

In [ ]:
from pathlib import Path
import json
import os
import re
from time import perf_counter

import faiss
import numpy as np
import pandas as pd
import pymupdf
from dotenv import load_dotenv
from google import genai
from google.genai import types
from IPython.display import display
from sentence_transformers import SentenceTransformer, CrossEncoder

PDF_PATH = Path("data/twilight_lma.pdf")
RESULTS_PATH = Path("results/rag_results.json")
load_dotenv(Path(".env"))
GEMINI_MODEL = os.getenv("GEMINI_MODEL", "gemini-3.5-flash-lite")
API_KEY = os.getenv("GEMINI_API_KEY", "").strip()

pages = []
with pymupdf.open(PDF_PATH) as document:
    for page_number, page in enumerate(document, start=1):
        midpoint = page.rect.width / 2
        blocks = [b for b in page.get_text("blocks") if b[4].strip() and b[6] == 0]
        # The central gutter separates the two columns on the rules pages.
        blocks.sort(key=lambda b: (b[0] >= midpoint, round(b[1], 1), b[0]))
        text = "\n".join(b[4].strip() for b in blocks)
        pages.append({"page": page_number, "text": text})

print(f"Extracted {len(pages)} PDF pages.")
for page_number in (4, 5):
    print(f"\nPDF page {page_number} preview:\n{pages[page_number - 1]['text'][:650]}\n")
assert pages[4]["text"].find("PLACE SYSTEM TILES") < pages[4]["text"].find("STEP 12")


## 2. Make overlapping, page-aware chunks

The embedding model tokenizer defines chunk boundaries. Each chunk stays within one PDF page, overlaps its neighbor by 40 tokens, and has a deterministic page/offset ID. We check every chunk against the model's token limit.

In [ ]:
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
RERANK_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"
embedder = SentenceTransformer(EMBEDDING_MODEL)
reranker = CrossEncoder(RERANK_MODEL)

CHUNK_TOKENS = min(180, embedder.max_seq_length - 2)
OVERLAP_TOKENS = 40
assert CHUNK_TOKENS > OVERLAP_TOKENS
chunks = []
for page in pages:
    encoded = embedder.tokenizer(page["text"], add_special_tokens=False,
                                 return_offsets_mapping=True, verbose=False)
    offsets = encoded["offset_mapping"]
    step = CHUNK_TOKENS - OVERLAP_TOKENS
    for start in range(0, len(offsets), step):
        end = min(start + CHUNK_TOKENS, len(offsets))
        # Token offsets keep the PDF's original wording and punctuation.
        text = page["text"][offsets[start][0]:offsets[end - 1][1]].strip()
        if text:
            assert len(embedder.tokenizer.encode(text)) <= embedder.max_seq_length
            chunks.append({"id": f"p{page['page']:02d}-t{start:05d}",
                           "page": page["page"], "text": text})
        if end >= len(offsets):
            break

assert len({chunk["id"] for chunk in chunks}) == len(chunks)
print(f"{len(chunks)} chunks; {CHUNK_TOKENS} tokens maximum, {OVERLAP_TOKENS} overlap.")
display(pd.DataFrame(chunks)[["id", "page", "text"]].head(3))


## 3. Embed and index once

Both document and query vectors are normalized. FAISS inner product then ranks them by cosine similarity. Loading models and building this index happen before per-query timers start.

In [ ]:
vectors = embedder.encode([chunk["text"] for chunk in chunks],
                          convert_to_numpy=True, normalize_embeddings=True,
                          show_progress_bar=True).astype("float32")
index = faiss.IndexFlatIP(vectors.shape[1])
index.add(vectors)
assert index.ntotal == len(chunks)
print(f"Indexed {index.ntotal} vectors of dimension {vectors.shape[1]}.")


## 4. Choose questions and known evidence

Edit `QUERIES` freely. `evidence` is a short phrase from the PDF that would support the answer; it is used only to check rank movement, never to retrieve or re-rank. Include exceptions, negations, and plausible distractors.

In [ ]:
QUERIES = [
    {"question": "Can trade goods be spent to cast votes during the agenda phase?",
     "evidence": "Trade goods cannot be spent to cast votes"},
    {"question": "May a player spend commodities directly, without trading them first?",
     "evidence": "A player cannot spend commodities"},
    {"question": "Can ships move through an asteroid field if they do not stop there?",
     "evidence": "A ship cannot move through or into an asteroid field"},
    {"question": "Can a player pass before using their strategy card?",
     "evidence": "A player cannot pass until they have performed a strategic"},
    {"question": "Can a blockaded space dock still produce ground forces but not ships?",
     "evidence": "A player cannot use a blockaded unit to produce ships"},
    {"question": "Can a unit without a printed cost be produced by an ability?",
     "evidence": "If a unit does not have a cost, it cannot be produced"},
]


## 5. Retrieve, re-rank, and answer

Each question gets exactly 10 FAISS candidates. The cross-encoder scores those same 10, with no new retrieval. Both answer calls use the same prompt function, model, settings, and context format; only the selected top three chunks differ. Timings exclude model loading and indexing.

In [ ]:
def format_context(selected):
    return "\n\n".join(
        f"[Chunk {item['id']} | PDF page {item['page']}]\n{item['text']}"
        for item in selected
    )


def answer(question, context, client):
    prompt = ("Answer the question using only the context below. If the context does not "
              "contain the answer, say you cannot determine it from the supplied context. "
              "Cite supporting PDF page numbers.\n\n"
              f"Question: {question}\n\nContext:\n{context}")
    response = client.models.generate_content(
        model=GEMINI_MODEL,
        contents=prompt,
        config=types.GenerateContentConfig(temperature=0, max_output_tokens=300),
    )
    return response.text or "[No text returned]"


client = genai.Client(api_key=API_KEY) if API_KEY else None
if client is None:
    print("GEMINI_API_KEY unavailable: answer generation is UNEXECUTED; retrieval and re-ranking will run.")

results = []
for query in QUERIES:
    question = query["question"]
    start = perf_counter()
    query_vector = embedder.encode([question], convert_to_numpy=True,
                                   normalize_embeddings=True).astype("float32")
    embedding_ms = (perf_counter() - start) * 1000

    start = perf_counter()
    similarities, positions = index.search(query_vector, 10)
    search_ms = (perf_counter() - start) * 1000
    candidates = []
    for rank, (position, similarity) in enumerate(zip(positions[0], similarities[0]), 1):
        if position < 0:
            continue
        candidates.append({**chunks[int(position)], "original_rank": rank,
                           "cosine_score": float(similarity)})

    start = perf_counter()
    rerank_scores = reranker.predict([(question, item["text"]) for item in candidates])
    rerank_ms = (perf_counter() - start) * 1000
    for item, score in zip(candidates, rerank_scores):
        item["rerank_score"] = float(score)
    reordered = sorted(candidates, key=lambda item: item["rerank_score"], reverse=True)
    for rank, item in enumerate(reordered, 1):
        item["reranked_rank"] = rank

    baseline_context = format_context(candidates[:3])
    reranked_context = format_context(reordered[:3])
    answers = {"baseline": None, "reranked": None}
    answer_errors = {"baseline": None, "reranked": None}
    generation_ms = {"baseline": None, "reranked": None}
    if client is not None:
        for name, context in (("baseline", baseline_context), ("reranked", reranked_context)):
            start = perf_counter()
            try:
                answers[name] = answer(question, context, client)
            except Exception as exc:
                answer_errors[name] = type(exc).__name__
            generation_ms[name] = (perf_counter() - start) * 1000

    result = {"question": question, "evidence": query["evidence"],
              "candidates": candidates, "baseline_context": baseline_context,
              "reranked_context": reranked_context, "answers": answers,
              "answer_errors": answer_errors,
              "latency_ms": {"query_embedding": embedding_ms, "vector_search": search_ms,
                             "reranking": rerank_ms, "baseline_generation": generation_ms["baseline"],
                             "reranked_generation": generation_ms["reranked"]}}
    results.append(result)
    print(f"\nQUESTION: {question}")
    display(pd.DataFrame(candidates)[["original_rank", "reranked_rank", "id", "page",
                                      "cosine_score", "rerank_score"]].round(3))
    print("BASELINE CONTEXT (vector top 3):\n", baseline_context)
    print("RERANKED CONTEXT (new top 3):\n", reranked_context)
    print("BASELINE ANSWER:", answers["baseline"] or
          (f"FAILED ({answer_errors['baseline']})" if client else "UNEXECUTED (missing API key)"))
    print("RERANKED ANSWER:", answers["reranked"] or
          (f"FAILED ({answer_errors['reranked']})" if client else "UNEXECUTED (missing API key)"))


## 6. Compare latency and inspect evidence movement

The table separates embedding, FAISS search, cross-encoder scoring, and each Gemini call. A successful movement is shown only when the supplied evidence phrase occurs in an actual candidate at vector rank 4–10 and in a candidate promoted to re-ranked rank 1–3. Check the displayed contexts before drawing conclusions.

In [ ]:
latency_rows = []
movement_rows = []
for result in results:
    times = result["latency_ms"]
    latency_rows.append({"question": result["question"], **times,
                         "baseline_total_ms": (times["query_embedding"] + times["vector_search"]
                                               + times["baseline_generation"]
                                               if times["baseline_generation"] is not None else None),
                         "reranked_total_ms": (times["query_embedding"] + times["vector_search"]
                                               + times["reranking"] + times["reranked_generation"]
                                               if times["reranked_generation"] is not None else None)})
    phrase = re.sub(r"\s+", " ", result["evidence"]).casefold()
    matches = [item for item in result["candidates"]
               if phrase in re.sub(r"\s+", " ", item["text"]).casefold()]
    promoted = [item for item in matches
                if 4 <= item["original_rank"] <= 10 and item["reranked_rank"] <= 3]
    movement_rows.append({"question": result["question"], "evidence": result["evidence"],
                          "evidence_found_in_top_10": bool(matches),
                          "movement_4_to_3": bool(promoted),
                          "verified_moves": [(item["id"], item["page"],
                                               item["original_rank"], item["reranked_rank"])
                                              for item in promoted]})

display(pd.DataFrame(latency_rows).round(1))
display(pd.DataFrame(movement_rows))
if not any(row["movement_4_to_3"] for row in movement_rows):
    print("No verified rank-4–10 to top-3 evidence movement in these queries. Edit QUERIES and rerun to explore further.")


## 7. Save the experiment

The JSON includes full candidate text, both contexts and answers, timings, and evidence checks. It is ignored by Git so repeated experiments stay local.

In [ ]:
RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)
payload = {"pdf": str(PDF_PATH), "embedding_model": EMBEDDING_MODEL,
           "rerank_model": RERANK_MODEL, "gemini_model": GEMINI_MODEL,
           "answer_generation": ("unexecuted_missing_api_key" if not client else
                                 "completed" if all(all(r["answers"].values()) for r in results)
                                 else "attempted_with_errors"),
           "chunk_tokens": CHUNK_TOKENS, "overlap_tokens": OVERLAP_TOKENS,
           "results": results, "latency_comparison": latency_rows,
           "evidence_movement": movement_rows}
RESULTS_PATH.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Saved {RESULTS_PATH}")
